# Lab 4 Exercise, part 2: a custom slide skill

This notebook solves the Lab 4 stretch challenge. It gives the slide-maker a Northwind Advisory house style instead of Voltway's.

The skill lives in a folder with `SKILL.md`. The top block names and describes it, while the markdown sets the writing rules.

The file is `sandbox/skills/northwind-slide/SKILL.md`. A sample comparison and slide tool let this notebook run on its own.

## Setup

Import the Deep Agent code and `python-pptx`. Write all files to the sandbox.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import MSO_ANCHOR
from pptx.util import Inches, Pt

load_dotenv(override=True)

sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

## Sample briefing

Write a short comparison so the notebook does not depend on an earlier agent run.

In [ ]:
comparison = '''# Fleet EV comparison: Tesla Model Y vs Ford Mustang Mach-E

## Tesla Model Y
- Real-world range around 330 miles
- Starting price around 45,000 dollars
- Access to the Supercharger fast charging network

## Ford Mustang Mach-E
- Real-world range around 300 miles
- Starting price around 43,000 dollars
- Uses public CCS fast charging, now with Supercharger access

## Recommendation
For a 100-car sales fleet, choose the Tesla Model Y for its range and charging network. The Mustang Mach-E is a close, cheaper runner-up.
'''

with open(os.path.join(sandbox, "comparison.md"), "w", encoding="utf-8") as f:
    f.write(comparison)

print("Wrote comparison.md into the sandbox")

## Northwind slide skill

The skill asks for a short claim title, three labeled points, and one clear reason for the choice. The skill controls the words; the tool controls the design.

Read the file below. The agent later finds it through the skills folder.

In [ ]:
skill_path = os.path.join(sandbox, "skills", "northwind-slide", "SKILL.md")
with open(skill_path, encoding="utf-8") as skill_file:
    print(skill_file.read())

## Northwind slide tool

The skill guides the words. `create_slide` applies a forest green and amber design with a Northwind footer.

In [ ]:
NORTHWIND = RGBColor(0x14, 0x3D, 0x2E)
AMBER = RGBColor(0xF2, 0xA6, 0x1C)
CREAM = RGBColor(0xF5, 0xF1, 0xE6)


def _textbox(slide, left, top, width, height, text, size, color, bold=False):
    box = slide.shapes.add_textbox(Inches(left), Inches(top), Inches(width), Inches(height))
    frame = box.text_frame
    frame.word_wrap = True
    run = frame.paragraphs[0].add_run()
    run.text = text
    run.font.size = Pt(size)
    run.font.color.rgb = color
    run.font.bold = bold
    return box


def build_slide(title, key_points, recommendation, outfile):
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    slide = prs.slides.add_slide(prs.slide_layouts[6])

    slide.background.fill.solid()
    slide.background.fill.fore_color.rgb = NORTHWIND

    _textbox(slide, 0.5, 0.5, 11, 0.5, 'NORTHWIND ADVISORY  |  FLEET INTELLIGENCE', 15, AMBER, bold=True)

    _textbox(slide, 0.5, 1.5, 12.3, 1.2, title, 34, CREAM, bold=True)
    rule = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0.53), Inches(2.55), Inches(2.2), Inches(0.05))
    rule.fill.solid()
    rule.fill.fore_color.rgb = AMBER
    rule.line.fill.background()

    for i, point in enumerate(key_points[:3]):
        y = 3.05 + i * 0.72
        marker = slide.shapes.add_shape(MSO_SHAPE.OVAL, Inches(0.55), Inches(y + 0.1), Inches(0.16), Inches(0.16))
        marker.fill.solid()
        marker.fill.fore_color.rgb = AMBER
        marker.line.fill.background()
        _textbox(slide, 0.95, y, 11.8, 0.6, point, 17, CREAM)

    _textbox(slide, 0.53, 5.55, 6, 0.4, 'NORTHWIND RECOMMENDS', 12, AMBER, bold=True)
    banner = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(0.5), Inches(5.95), Inches(12.33), Inches(1.05))
    banner.adjustments[0] = 0.18
    banner.fill.solid()
    banner.fill.fore_color.rgb = AMBER
    banner.line.fill.background()
    frame = banner.text_frame
    frame.word_wrap = True
    frame.vertical_anchor = MSO_ANCHOR.MIDDLE
    frame.margin_left = Inches(0.35)
    frame.margin_right = Inches(0.35)
    run = frame.paragraphs[0].add_run()
    run.text = recommendation
    run.font.size = Pt(16)
    run.font.bold = True
    run.font.color.rgb = NORTHWIND

    prs.save(outfile)


@tool
def create_slide(title: str, key_points: list[str], recommendation: str) -> str:
    """Create a one-slide PowerPoint in the Northwind Advisory house style, saved as northwind_fleet.pptx."""
    build_slide(title, key_points, recommendation, os.path.join(sandbox, "northwind_fleet.pptx"))
    return "Saved the slide to /northwind_fleet.pptx"

## Slide-maker and presenter

The slide-maker gets `create_slide` and `skills=["/skills/"]`. The presenter reads the comparison and sends the slide task to it.

In [ ]:
slide_maker = {
    "name": "slide-maker",
    "description": "Turns a finished recommendation into a one-slide PowerPoint deck.",
    "system_prompt": "You turn research recommendations into slides, following your northwind-slide skill.",
    "tools": [create_slide],
    "skills": ["/skills/"],
}

presenter = create_deep_agent(
    model=ChatOpenAI(model="gpt-5.4-mini"),
    subagents=[slide_maker],
    system_prompt="You prepare research for presentation by delegating to your slide-maker sub-agent.",
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

## Run it

The presenter reads `comparison.md`. The slide-maker reads the Northwind skill and calls `create_slide`.

In [ ]:
result = presenter.invoke({"messages": [{"role": "user", "content":
    "Read comparison.md and have a one-slide deck made of its recommendation."}]})
print(result["messages"][-1].content)

Open `northwind_fleet.pptx` in the sandbox. Change `SKILL.md` to change the writing rules, or `build_slide` to change the design.